# Day 1: Infrastructure Foundation & Iceberg Storage Setup

This notebook simulates the local environment for Apache Iceberg catalogs and creates the Bronze raw payload schema with audit metadata.

In [ ]:
!pip install pyspark pyiceberg duckdb splink -q

In [ ]:
import os
from pyspark.sql import SparkSession

# Initialize Spark Session with Iceberg configurations
spark = SparkSession.builder \
    .appName("Day01_Storage_Setup") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.4_2.12:1.4.1") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "/tmp/iceberg_warehouse") \
    .getOrCreate()

print("Spark Session initialized with Iceberg catalog.")

In [ ]:
# Create the Bronze raw payload schema in the local mock catalog
spark.sql("""
CREATE DATABASE IF NOT EXISTS local.ehdip_bronze_db
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS local.ehdip_bronze_db.raw_payloads (
    ingestion_timestamp TIMESTAMP,
    source_system_id STRING,
    payload_id STRING,
    raw_payload_json STRING
)
USING iceberg
PARTITIONED BY (days(ingestion_timestamp))
""")

print("Bronze schema created successfully.")
spark.sql("DESCRIBE TABLE local.ehdip_bronze_db.raw_payloads").show(truncate=False)